PACE Stage: Plan
Project Background

This project analyzes historical YouTube Shorts data from Dr. Agustin Landivar's channel. The goal is to transform performance metrics into useful recommendations for content planning and business decision-making.

Business Problem

The organization needs to understand which videos and content characteristics are associated with stronger results in views, audience growth, click-through rate, and estimated revenue.

Primary Business Question

Which characteristics of YouTube Shorts are associated with stronger performance, and how can these findings improve the channel's content strategy?

Stakeholders
Dr. Agustin Landivar — CEO: Strategic content and channel growth decisions.
Content team: Selection of topics and future video ideas.
Video editors: Decisions related to video duration and viewing behavior.
Graphic designers: Visual decisions that may be evaluated in a future phase with additional data.
E-commerce team: Alignment of popular health topics with content for 

- [doctorlandivar.com](https://doctorlandivar.com)
- [finelandvitamins.com](https://finelandvitamins.com)

Key Analytical Questions

What is the overall performance of the analyzed YouTube Shorts?
Which videos generated the most and least views, estimated revenue, and new users?
How are views, impressions, click-through rate, and estimated revenue related?
How does video duration relate to performance and average view duration?
How does performance vary by publication date, month, and day of the week?
What is the distribution of new, recurring, occasional, and regular users?
Which words, topics, or title characteristics appear most frequently among high-performing videos?
Are there outliers that could distort averages or business conclusions?

Deliverables

Cleaned and anonymized dataset
Exploratory data analysis with Python
Business analysis with SQL
Tableau dashboard
Interactive web presentation
Documented GitHub repository
Data-driven recommendations
Limitations and Privacy

This analysis can identify associations but cannot prove causality.

The dataset does not contain website traffic, product transactions, thumbnail characteristics, or direct sales conversions. Therefore, it cannot determine whether a specific video caused product sales.

The original Excel file contains private business information and will not be published. Only anonymized or aggregated data will be included in the public portfolio.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Identify the project root regardless of the notebook's working directory

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_data_directory = project_root / "data" / "raw"
excel_files = list(raw_data_directory.glob("*.xlsx"))

print(f"Found root: {project_root.resolve()}")
print(f"Excel files found: {len(excel_files)}")
print(excel_files)

Found root: C:\Users\USUARIO1\Documents\youtube-shorts-performance-analysis
Excel files found: 1
[WindowsPath('c:/Users/USUARIO1/Documents/youtube-shorts-performance-analysis/data/raw/TODOS LOS SHORTS CON DATOS.xlsx')]


In [3]:
data_path = excel_files[0]
excel_workbook = pd.ExcelFile(data_path)
print(excel_workbook.sheet_names)

['Datos de la tabla', 'Datos del gráfico', 'Totales']


## PACE Stage: Analyze

### 1. Load the Raw Data

The main worksheet is loaded into a Pandas DataFrame. The raw dataset will remain unchanged so that every cleaning and transformation step can be documented and reproduced.

In [4]:
videos_raw = pd.read_excel(
    data_path,
    sheet_name="Datos de la tabla"
)

print(f"Rows: {videos_raw.shape[0]}")
print(f"Columns: {videos_raw.shape[1]}")

videos_raw.head()

Rows: 347
Columns: 13


,Contenido,Título del video,Tiempo de publicación del video,Duración,Usuarios nuevos,Usuarios recurrentes,Usuarios ocasionales,Usuarios habituales,Ingresos estimados (USD),Vistas,Impresiones,Tasa de clics de las impresiones (%),Duración promedio de vistas
0,Total,NaN,NaN,NaN,6594842,13306606,10717640,2588966,7582.878,138204991,241768987,4.71,0:01:08
1,zIXXepy6XR8,"Transforma Tu Noche, Transforma Tu Cuerpo 🌙 #b...","May 8, 2026",60.0,13508,290856,126914,163942,52.431,854603,58762,4.23,0:01:00
2,E2UQhp1fGwE,El secreto del colágeno que nadie te contó #co...,"May 6, 2026",77.0,192,25511,5573,19938,3.521,67564,38325,4.56,0:01:22
3,sBAzGcD35Dw,¿Dulce Veneno? El endulzante que alimenta al c...,"May 4, 2026",85.0,304,8988,2107,6881,1.306,26266,31679,3.79,0:01:24
4,JBT76wx6Eb0,¡El Secreto de la Eterna Juventud! ✨ Ácido Hia...,"Apr 29, 2026",91.0,225,18537,4015,14522,2.489,47497,58766,5.15,0:01:30


### 2. Inspect the Dataset Structure

The dataset structure is inspected before applying any transformation. This step identifies column names, data types, non-null values, and potential data quality issues.

In [5]:
videos_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 347 entries, 0 to 346
Data columns (total 13 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Contenido                             347 non-null    str    
 1   Título del video                      346 non-null    str    
 2   Tiempo de publicación del video       346 non-null    str    
 3   Duración                              346 non-null    float64
 4   Usuarios nuevos                       347 non-null    int64  
 5   Usuarios recurrentes                  347 non-null    int64  
 6   Usuarios ocasionales                  347 non-null    int64  
 7   Usuarios habituales                   347 non-null    int64  
 8   Ingresos estimados (USD)              347 non-null    float64
 9   Vistas                                347 non-null    int64  
 10  Impresiones                           347 non-null    int64  
 11  Tasa de clics de las impresion

### 3. Audit Missing Values

Missing values are quantified before data cleaning to determine whether they represent data quality problems or expected summary-row behavior.

In [6]:
data_quality = pd.DataFrame({
    "data_type": videos_raw.dtypes.astype(str),
    "missing_values": videos_raw.isna().sum(),
    "missing_percentage": (videos_raw.isna().mean() * 100).round(2)
})

data_quality = data_quality.reset_index()
data_quality = data_quality.rename(columns={"index":"column"})

data_quality


,column,data_type,missing_values,missing_percentage
0,Contenido,str,0,0.00
1,Título del video,str,1,0.29
2,Tiempo de publicación del video,str,1,0.29
3,Duración,float64,1,0.29
4,Usuarios nuevos,int64,0,0.00
5,Usuarios recurrentes,int64,0,0.00
6,Usuarios ocasionales,int64,0,0.00
7,Usuarios habituales,int64,0,0.00
8,Ingresos estimados (USD),float64,0,0.00
9,Vistas,int64,0,0.00


### 4. Audit Duplicate Records

Duplicate rows and repeated video identifiers are checked before cleaning the dataset. Duplicate records could distort totals, averages, and business conclusions.

In [7]:
duplicates_row = videos_raw.duplicated().sum()
duplicates_video_ids = videos_raw["Contenido"].duplicated().sum()

print(f"Duplicate rows: {duplicates_row}")
print(f"Duplicate video IDs: {duplicates_video_ids}")


Duplicate rows: 0
Duplicate video IDs: 0


### 5. Separate the Summary Row

The YouTube export contains a summary row labeled `Total`. This row is stored separately because it represents aggregated channel metrics rather than an individual video. A clean working DataFrame is then created with video-level records only.

In [8]:
summary_rows = videos_raw.loc[
    videos_raw["Contenido"] == "Total"
].copy()

videos_clean = videos_raw.loc[
    videos_raw["Contenido"] != "Total"
].copy()

videos_clean.reset_index(drop=True, inplace=True)

print(f"Summary rows stored: {len(summary_rows)}")
print(f"Video rows retained: {len(videos_clean)}")

videos_clean.head(3)

Summary rows stored: 1
Video rows retained: 346


,Contenido,Título del video,Tiempo de publicación del video,Duración,Usuarios nuevos,Usuarios recurrentes,Usuarios ocasionales,Usuarios habituales,Ingresos estimados (USD),Vistas,Impresiones,Tasa de clics de las impresiones (%),Duración promedio de vistas
0,zIXXepy6XR8,"Transforma Tu Noche, Transforma Tu Cuerpo 🌙 #b...","May 8, 2026",60.0,13508,290856,126914,163942,52.431,854603,58762,4.23,0:01:00
1,E2UQhp1fGwE,El secreto del colágeno que nadie te contó #co...,"May 6, 2026",77.0,192,25511,5573,19938,3.521,67564,38325,4.56,0:01:22
2,sBAzGcD35Dw,¿Dulce Veneno? El endulzante que alimenta al c...,"May 4, 2026",85.0,304,8988,2107,6881,1.306,26266,31679,3.79,0:01:24


### 6. Standardize Column Names

The original Spanish column names are preserved in `videos_raw`. The working dataset uses descriptive English names in `snake_case` format to improve readability, consistency, and reproducibility.

In [13]:
column_mapping = {
    "Contenido": "video_id",
    "Título del video": "video_title",
    "Tiempo de publicación del video": "publish_date",
    "Duración": "video_duration_seconds",
    "Usuarios nuevos": "new_viewers",
    "Usuarios recurrentes": "returning_viewers",
    "Usuarios ocasionales": "casual_viewers",
    "Usuarios habituales": "regular_viewers",
    "Ingresos estimados (USD)": "estimated_revenue_usd",
    "Vistas": "views",
    "Impresiones": "impressions",
    "Tasa de clics de las impresiones (%)": "impressions_ctr_percent",
    "Duración promedio de vistas": "average_view_duration"
}

videos_clean = videos_clean.rename(columns=column_mapping)

videos_clean.columns.tolist()



['video_id',
 'video_title',
 'publish_date',
 'video_duration_seconds',
 'new_viewers',
 'returning_viewers',
 'casual_viewers',
 'regular_viewers',
 'estimated_revenue_usd',
 'views',
 'impressions',
 'impressions_ctr_percent',
 'average_view_duration']

### 7. Convert Publication Dates

The publication date is converted from text into a datetime data type. This enables chronological analysis and the creation of time-based features such as year, month, and day of the week.

In [14]:
videos_clean["publish_date"] = pd.to_datetime(
    videos_clean["publish_date"],
    format="mixed",
    errors="coerce"
)

print(f"Data type : {videos_clean['publish_date'].dtype}")
print(f"Invalid dates: {videos_clean['publish_date'].isna().sum()}")
print(f"Earliest publish date: {videos_clean['publish_date'].min()}")
print(f"Latest publish date: {videos_clean['publish_date'].max()}")

Data type : datetime64[us]
Invalid dates: 0
Earliest publish date: 2023-05-31 00:00:00
Latest publish date: 2026-05-08 00:00:00


### 8. Convert Average View Duration to Seconds

The average view duration is stored as text in `hours:minutes:seconds` format. It is converted into total seconds to enable numerical analysis and comparisons with video duration.

In [15]:
average_view_timedelta = pd.to_timedelta(
    videos_clean["average_view_duration"],
    errors="coerce"
)

videos_clean["average_view_duration_seconds"] = (
    average_view_timedelta.dt.total_seconds()
)

print(
    f"Invalid durations: "
    f"{videos_clean['average_view_duration_seconds'].isna().sum()}"
)

videos_clean[
    [
        "average_view_duration",
        "average_view_duration_seconds"
    ]
].head()




Invalid durations: 0


,average_view_duration,average_view_duration_seconds
0,0:01:00,60.0
1,0:01:22,82.0
2,0:01:24,84.0
3,0:01:30,90.0
4,0:00:45,45.0


### 9. Review Descriptive Statistics

Descriptive statistics are calculated for the numerical variables to understand their distributions, typical values, variability, and potential outliers.

In [16]:
numeric_columns = [
    "video_duration_seconds",
    "new_viewers",
    "returning_viewers",
    "casual_viewers",
    "regular_viewers",
    "estimated_revenue_usd",
    "views",
    "impressions",
    "impressions_ctr_percent",
    "average_view_duration_seconds"
]

numeric_summary = (
    videos_clean[numeric_columns]
    .describe()
    .transpose()
    .round(2)
)

numeric_summary


,count,mean,std,min,25%,50%,75%,max
video_duration_seconds,346.0,65.60,23.15,9.00,54.00,60.00,73.75,173.00
new_viewers,346.0,16798.74,47958.77,70.00,610.00,1833.50,9313.50,583832.00
returning_viewers,346.0,199107.66,324751.78,4858.00,33889.25,78784.00,205879.75,2312814.00
casual_viewers,346.0,162085.15,316393.71,1524.00,9379.00,37938.50,152922.00,2312613.00
regular_viewers,346.0,37022.51,47461.99,71.00,8057.75,20850.00,44857.00,258750.00
estimated_revenue_usd,346.0,19.15,28.75,0.71,4.19,8.07,20.04,184.77
views,346.0,347234.78,513511.77,10841.00,79732.75,156378.50,351753.75,3352263.00
impressions,346.0,671259.86,2171445.02,21585.00,88510.75,162364.50,540957.25,29415236.00
impressions_ctr_percent,346.0,4.85,0.94,2.02,4.34,4.92,5.50,7.85
average_view_duration_seconds,346.0,67.45,17.87,22.00,56.00,67.00,79.00,144.00


#### Initial Observations

- All numerical variables contain 346 valid records.
- Views, impressions, revenue, and viewer metrics have means considerably higher than their medians, suggesting right-skewed distributions and potential high-performing outliers.
- The impressions click-through rate is more stable, with a mean of 4.85% and a median of 4.92%.
- The median video duration is 60 seconds.
- Additional validation is required before unusual values are classified as errors or legitimate outliers.

### 10. Validate Numerical Business Rules

Numerical variables are checked for impossible values and internal inconsistencies before exploratory visualizations are created.

In [18]:
validation_results = {
    "negative_numeric_values":int(
        (videos_clean[numeric_columns] <= 0).sum().sum()
    ),
    "non_positive_video_duration":int(
        (videos_clean["video_duration_seconds"] <= 0).sum()
    ),
    "invalid_ctr_values": int(
        (
            ~videos_clean["impressions_ctr_percent"].between(0, 100)
        ).sum()
    )
}

pd.Series(validation_results, name="issues_found")



negative_numeric_values        0
non_positive_video_duration    0
invalid_ctr_values             0
Name: issues_found, dtype: int64